# Task 1 — Notebook 02: Multi-year NDVI phenology (corn vs soybean, CDL masks)

This notebook **replaces** the older single-year exploratory flow. It:

1. **Discovers calendar years** where you have **both** a CDL label column (`cdl_{year}` in `cdl_stack_wide.parquet`) **and** a matching NDVI weekly wide file (`ndvi_weekly_{year}_wide.parquet`). With your current pipeline this is typically **2008–2025** (18 years of CDL columns × NDVI; NDVI on disk may extend to 2000+ but CDL starts at 2008).
2. For **each year**, merges CDL + NDVI on `(iy, ix)`, applies **hard corn/soy masks** (codes 1 and 5), and summarizes **weekly mean NDVI** and **pixel-wise IQR** bands (spatial spread at each week index).
3. Saves a **tidy table** for reporting and plots **multi-year** curves.

**Run notebook 01 first** if you need to refresh interim/processed data or verify schemas.

**Literature / QA:** Your `resources/` PDFs (MODIS user guide; RS&E / Remote Sensing of Environment time-series VI papers; Ecography, etc.) are the authority for **composite timing, scaling, smoothing, and phenology metric definitions**. This notebook does not embed those PDFs; use them as **sanity checks** (expected NDVI range, unimodal growing-season shape for corn/soy in the Corn Belt, SOS < peak < EOS). Cite the papers you actually read in the final report.


## Methods notes (align with your PDFs — sanity checklist)

Use the following as a **checklist** while you read `resources/*.pdf` (filenames you listed):

| Check | Why it matters |
|-------|----------------|
| **VI valid range / scale** | MOD13 / product guides specify valid DN or physical NDVI; outliers or fill values should be filtered before interpreting means. |
| **Composite date / week index** | Weekly `w000…` are **ordinal bands** for that file; map to real dates via `ndvi_weekly_{year}_metadata.json` → `time_start_day` (compare to MODIS composite-day logic in the user guide). |
| **Unimodal season** | For a single summer crop, NDVI often rises, peaks, then falls; weird bimodality may indicate clouds, double-cropping, or mis-registration. |
| **CDL as categorical mask** | CDL is **not** ground truth at 250 m; label noise and mixed pixels bias simple `cdl==1` masks. Your YAML `purity_filtering` is the stricter approach for a later notebook. |
| **Inter-annual variability** | Weather and planting dates shift curves year-to-year; comparing **multiple years** (this notebook) is more informative than a single season.

*Do not cite a paper in the course report unless the claim is supported by what you read in that paper.*


In [ ]:
# --- Setup: paths, config, imports ---
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root (requirements.txt + src/). Open Jupyter from the repo or cd into it.")
sys.path.insert(0, str(REPO_ROOT))

CFG_PATH = REPO_ROOT / "configs" / "task1_ndvi_analysis.yaml"
with open(CFG_PATH, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

CORN = int(cfg["cdl"]["crop_classes"]["corn"])
SOY = int(cfg["cdl"]["crop_classes"]["soybean"])
PHCFG = cfg.get("phenology_years") or {}
YMIN, YMAX = PHCFG.get("year_min"), PHCFG.get("year_max")
EXPLICIT = PHCFG.get("explicit")  # optional list

print("Corn / soy CDL codes:", CORN, SOY)
print("phenology_years from YAML:", {k: PHCFG.get(k) for k in ("year_min", "year_max", "explicit") if k in PHCFG or k == "explicit"})


In [ ]:
def discover_phenology_years(repo: Path) -> list[int]:
    """Calendar years with both CDL column and NDVI wide Parquet on disk."""
    cdl_pq = repo / "data" / "processed" / "cdl" / "cdl_stack_wide.parquet"
    if not cdl_pq.is_file():
        raise FileNotFoundError(cdl_pq)
    names = pq.ParquetFile(cdl_pq).schema_arrow.names
    cdl_years = sorted(int(n.replace("cdl_", "")) for n in names if n.startswith("cdl_"))

    ndvi_dir = repo / "data" / "processed" / "ndvi"
    ndvi_years: set[int] = set()
    for p in ndvi_dir.glob("ndvi_weekly_*_wide.parquet"):
        stem = p.stem  # ndvi_weekly_2022_wide
        stem = stem.replace("ndvi_weekly_", "").replace("_wide", "")
        ndvi_years.add(int(stem))

    years = sorted(y for y in cdl_years if y in ndvi_years)
    if EXPLICIT:
        years = sorted(y for y in EXPLICIT if y in ndvi_years and y in cdl_years)
    else:
        if YMIN is not None:
            years = [y for y in years if y >= int(YMIN)]
        if YMAX is not None:
            years = [y for y in years if y <= int(YMAX)]
    return years


YEARS = discover_phenology_years(REPO_ROOT)
print(f"Years for analysis (n={len(YEARS)}): {YEARS[0]} … {YEARS[-1]}" if YEARS else "No overlapping years!")
print("Full list:", YEARS)


## Step 1 — Load CDL columns once (all analysis years)

We only need `iy`, `ix`, and `cdl_{year}` for years in `YEARS`. Keeping one Parquet read avoids re-loading CDL for every loop iteration.


In [ ]:
cdl_path = REPO_ROOT / "data" / "processed" / "cdl" / "cdl_stack_wide.parquet"
cdl_cols = ["iy", "ix"] + [f"cdl_{y}" for y in YEARS]
cdl_df = pd.read_parquet(cdl_path, columns=cdl_cols)
print("CDL rows:", f"{len(cdl_df):,}", "| columns:", len(cdl_df.columns))
display(cdl_df.head(3))


## Step 2 — Per-year NDVI: merge, mask, weekly summary

For each calendar year:

1. Read `ndvi_weekly_{year}_wide.parquet`.
2. **Melt** wide week columns `w000…` → long `(iy, ix, week_index, ndvi)`.
3. Merge CDL for that year on `(iy, ix)`.
4. Split **corn** vs **soybean** pixels; **groupby week** → mean and 25/75% quantiles (IQR band across pixels).

*Performance:* one NDVI year at a time limits RAM versus melting all years at once. With many calendar years, the next cell can take **several minutes**.


In [ ]:
def load_ndvi_time_metadata(year: int) -> dict | None:
    meta_path = REPO_ROOT / "data" / "processed" / "ndvi" / f"ndvi_weekly_{year}_metadata.json"
    if not meta_path.is_file():
        return None
    return json.loads(meta_path.read_text(encoding="utf-8"))


def weekly_stats_for_year(year: int, cdl_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Returns (corn_stats, soy_stats, ndvi_value_summary) for one calendar year."""
    ndvi_path = REPO_ROOT / "data" / "processed" / "ndvi" / f"ndvi_weekly_{year}_wide.parquet"
    if not ndvi_path.is_file():
        raise FileNotFoundError(ndvi_path)

    ndvi = pd.read_parquet(ndvi_path)
    wcols = [c for c in ndvi.columns if c.startswith("w")]
    long = ndvi.melt(id_vars=["iy", "ix"], value_vars=wcols, var_name="band", value_name="ndvi")
    long["week"] = long["band"].str.replace("w", "", regex=False).astype(int)

    cdl_y = cdl_frame[["iy", "ix", f"cdl_{year}"]].rename(columns={f"cdl_{year}": "crop_code"})
    m = long.merge(cdl_y, on=["iy", "ix"], how="inner")

    corn = m[m["crop_code"] == CORN]
    soy = m[m["crop_code"] == SOY]

    def _stats(sub: pd.DataFrame, prefix: str) -> pd.DataFrame:
        g = sub.groupby("week", sort=True)["ndvi"]
        out = pd.DataFrame({
            f"{prefix}_mean": g.mean(),
            f"{prefix}_q25": g.quantile(0.25),
            f"{prefix}_q75": g.quantile(0.75),
            f"{prefix}_n_pix": sub.groupby("week").size(),
        }).reset_index()
        return out

    corn_s = _stats(corn, "corn")
    soy_s = _stats(soy, "soy")
    summ = {
        "year": year,
        "ndvi_min": float(m["ndvi"].min()),
        "ndvi_max": float(m["ndvi"].max()),
        "ndvi_median": float(m["ndvi"].median()),
        "n_week_cols": len(wcols),
    }
    return corn_s, soy_s, summ


rows_summ: list[dict] = []
all_curve_parts: list[pd.DataFrame] = []

for y in YEARS:
    print(f"… {y} ({YEARS.index(y)+1}/{len(YEARS)})")
    c_stats, s_stats, summ = weekly_stats_for_year(y, cdl_df)
    rows_summ.append(summ)
    merged = pd.merge(c_stats, s_stats, on="week", how="outer")
    merged.insert(0, "year", y)
    all_curve_parts.append(merged)

curves = pd.concat(all_curve_parts, ignore_index=True)
ndvi_sanity = pd.DataFrame(rows_summ).sort_values("year")
print("Per-year NDVI value sanity (min / max / median over all merged pixels that week):")
display(ndvi_sanity)


## Step 3 — Optional: attach real calendar dates to week indices

`time_start_day` in each year's NDVI metadata JSON aligns `w000` → first date, etc. Use this when comparing to **DOY** windows in `task1_ndvi_analysis.yaml` (`growing_season_doy`) or to MODIS composite documentation.


In [ ]:
def week_index_to_date_str(year: int, week_index: int) -> str | None:
    meta = load_ndvi_time_metadata(year)
    if not meta:
        return None
    tsd = meta.get("time_start_day") or []
    if week_index < 0 or week_index >= len(tsd):
        return None
    v = tsd[week_index]
    return str(v)[:10] if v is not None else None


sample = curves.loc[curves["year"] == YEARS[len(YEARS) // 2]].head(5).copy()
y_mid = int(sample["year"].iloc[0])
sample["approx_date"] = sample["week"].apply(lambda w: week_index_to_date_str(y_mid, int(w)))
print("Example week → calendar date:", y_mid)
display(sample[["year", "week", "approx_date", "corn_mean", "soy_mean"]])


## Step 4 — Plots (multi-year)

**Panel A:** mean corn NDVI vs week index, one line per year.  
**Panel B:** same for soybean.  
Week indices are **0-based band order** for that year's file, comparable across years only if each year's stack uses the same compositing rule (check metadata lengths — they can differ slightly).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
cmap = plt.cm.viridis(np.linspace(0.15, 0.95, len(YEARS)))

for ax, crop, col in zip(axes, ["Corn", "Soybean"], ["corn_mean", "soy_mean"]):
    for i, y in enumerate(YEARS):
        sub = curves[curves["year"] == y]
        ax.plot(sub["week"], sub[col], marker="o", ms=3, lw=1.2, color=cmap[i], label=str(y))
    ax.set_title(f"{crop} — mean NDVI by week index")
    ax.set_xlabel("Week index (w000 → 0)")
    ax.set_ylabel("NDVI")
    ax.grid(True, ls="--", alpha=0.4)

axes[1].legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap: corn mean NDVI (years × week) — quick view of inter-annual structure
piv = curves.pivot_table(index="year", columns="week", values="corn_mean")
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(piv.values, aspect="auto", interpolation="nearest", cmap="YlGn")
ax.set_yticks(range(len(piv.index)))
ax.set_yticklabels(piv.index)
ax.set_xticks(range(0, piv.shape[1], max(1, piv.shape[1] // 12)))
ax.set_xticklabels([piv.columns[i] for i in range(0, piv.shape[1], max(1, piv.shape[1] // 12))])
ax.set_xlabel("Week index")
ax.set_ylabel("Year")
ax.set_title("Corn — mean NDVI (pixel mean per week)")
plt.colorbar(im, ax=ax, label="NDVI")
plt.tight_layout()
plt.show()


## Step 5 — Export tidy table (optional)

Writes under `artifacts/tables/task1/` from config so graders can find outputs without re-running plots.


In [ ]:
out_dir = REPO_ROOT / (cfg.get("output") or {}).get("tables_dir", "artifacts/tables/task1/")
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "multi_year_corn_soy_weekly_ndvi_summary.csv"
curves.to_csv(out_csv, index=False)
ndvi_sanity.to_csv(out_dir / "multi_year_ndvi_value_sanity.csv", index=False)
print("Wrote:", out_csv.relative_to(REPO_ROOT))


## Caveats & next steps

- **Hard CDL mask** only — implement **purity / dominant fraction** (YAML `purity_filtering`) in a dedicated notebook to reduce mixed-pixel bias (`context/RISKS.md`).
- **Smoothing & phenometrics** (`smoothing`, `phenometrics` in YAML) belong in notebook 03+ per repo plan; compare extracted SOS/peak/EOS to literature curves.
- **SMAP** (2015+) is optional for Task 1; do not merge SMAP here unless resampled to the same `(iy, ix)` grid with documented alignment.

When `resources/*.pdf` are available locally, add **1–2 sentences per figure** tying your plots to specific guidance from MOD13 / RSE time-series papers (valid range, smoothing, phenology definition).
